# 🚀 50M Bengali GPT - ২-স্টেজ প্রোডাকশন ট্রেনিং (Free Colab Optimized)
### 🧠 ২-স্টেজ আর্কিটেকচার:
1. **স্টেজ ১ (Pretraining):** `corpus.txt` (~**১০ লাখ লাইন**, ~৩ কোটি টোকেন) — ২ ইপক, lr=3e-4 → মোট **~৬,৫৯০ স্টেপ** (~৩১ মিনিট)
2. **স্টেজ ২ (SFT Chatbot):** `sft_data.txt` (~**২ লাখ লাইন**) — ৩ ইপক, lr=1e-4 → মোট **~২২০ স্টেপ** (~৩ মিনিট)

### 📊 স্টেপ ক্যালকুলেশন (Step Math):
```
Tokens per Step = batch(4) × grad_accum(4) × context(512) = 8,192
Train Tokens    = 10L lines × 30 tok/line × 90% split   = ~2.7 কোটি
Steps per Epoch = 2.7 কোটি ÷ 8,192                     ≈ 3,295 steps
Total (2 epoch) = 3,295 × 2                             = ~6,590 steps
```
### ⚡ সর্বোচ্চ গতি ও সুরক্ষা (PyTorch 2.0):
- **PyTorch Speed:** SDPA FlashAttention + AMP FP16 + PyTorch 2.0 `torch.compile`
- **Auto-Resume:** সেশন কেটে গেলে শেষ স্টেপ থেকে স্বয়ংক্রিয় পুনরারম্ভ
- **কোনো LoRA/QLoRA নেই** — ১০০% আনফ্রোজেন Full Model Training
- **Context:** 512 tokens (~৩০০-৩৫০ বাংলা শব্দ), **Vocab:** 10,000 ByteLevel BPE
- **VRAM:** ~১.৮ GB (Free Colab T4-এ নিরাপদ)

In [ ]:
# Step 1: GPU চেক করুন (NVIDIA T4 থাকা নিশ্চিত করুন)
!nvidia-smi

In [ ]:
# Step 2: Google Drive মাউন্ট করুন
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
STAGE1_DIR = os.path.join(DRIVE_BASE_DIR, 'stage1_pretrain')
STAGE2_DIR = os.path.join(DRIVE_BASE_DIR, 'stage2_sft')
os.makedirs(STAGE1_DIR, exist_ok=True)
os.makedirs(STAGE2_DIR, exist_ok=True)
print(f"✓ Drive রেডি:\n  Stage 1: {STAGE1_DIR}\n  Stage 2: {STAGE2_DIR}")

In [ ]:
# Step 3: রিপোজিটরি ক্লোন ও ডিপেনডেন্সি ইনস্টল
%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git
%cd /content/ss_100m/ss_50million
!pip install -q -r requirements.txt
print("✓ পরিবেশ প্রস্তুত!")

In [ ]:
# Step 4: 📁 বিশাল ডেটাসেট প্রস্তুতকরণ
# Corpus target: ~১০ লাখ লাইন | SFT target: ~২ লাখ লাইন
import os, re, json, random
from datasets import load_dataset

os.makedirs('data', exist_ok=True)
corpus_path   = 'data/corpus.txt'
sft_data_path = 'data/sft_data.txt'

# স্কিপ কন্ডিশন: ফাইলে ৯ লাখের বেশি লাইন থাকলে পুনরায় ডাউনলোড এড়ানো হবে
need_download = True
if os.path.exists(corpus_path) and os.path.exists(sft_data_path):
    with open(corpus_path, 'r', encoding='utf-8') as f:
        c_cnt = sum(1 for _ in f)
    with open(sft_data_path, 'r', encoding='utf-8') as f:
        s_cnt = sum(1 for _ in f)
    if c_cnt >= 900000 and s_cnt >= 150000:
        print(f"✓ বিশাল কর্পাস বিদ্যমান (Corpus: {c_cnt:,} লাইন | SFT: {s_cnt:,} লাইন) — স্কিপ।")
        need_download = False
    else:
        print(f"⚠️ বিদ্যমান: Corpus={c_cnt:,}, SFT={s_cnt:,} — ১০ লাখ টার্গেটের জন্য রিডাউনলোড শুরু হচ্ছে...")

if need_download:
    print("=" * 65)
    print("📥 ১০ লাখ লাইনের বিশাল কর্পাস সংগ্রহ শুরু হচ্ছে...")
    print("=" * 65)
    corpus_lines = []
    sft_lines    = []

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # CORPUS SOURCE ১: বাংলা উইকিপিডিয়া (৫,০০,০০০ লাইন)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("  [১/৬] বাংলা উইকিপিডিয়া → ৫,০০,০০০ লাইন...")
    wiki_bn = load_dataset('wikimedia/wikipedia', '20231101.bn', split='train', streaming=True)
    bn_cnt = 0
    for item in wiki_bn:
        for p in item.get('text', '').split('\n'):
            p = p.strip()
            if len(p) >= 25 and re.search(r'[\u0980-\u09FF]', p):
                corpus_lines.append(p)
                bn_cnt += 1
                if bn_cnt >= 500000: break
        if bn_cnt >= 500000: break
    print(f"     ✓ বাংলা উইকিপিডিয়া: {bn_cnt:,} লাইন")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # CORPUS SOURCE ২: বাংলা সংবাদপত্র (Bangla Newspaper) ২,০০,০০০ লাইন
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("  [২/৬] বাংলা সংবাদপত্র → ২,০০,০০০ লাইন...")
    try:
        news_ds = load_dataset('zabir-nabil/bangla_newspaper_dataset', split='train', streaming=True)
        news_cnt = 0
        for row in news_ds:
            txt = (row.get('text', '') or row.get('content', '') or '').strip()
            for p in txt.split('\n'):
                p = p.strip()
                if len(p) >= 25 and re.search(r'[\u0980-\u09FF]', p):
                    corpus_lines.append(p)
                    news_cnt += 1
                    if news_cnt >= 200000: break
            if news_cnt >= 200000: break
        print(f"     ✓ বাংলা সংবাদপত্র: {news_cnt:,} লাইন")
    except Exception as e:
        print(f"     ⚠️ বাংলা সংবাদপত্র ফলব্যাক: {e}")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # CORPUS SOURCE ৩: ইংরেজি উইকিপিডিয়া (১,০০,০০০ লাইন)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("  [৩/৬] ইংরেজি উইকিপিডিয়া → ১,০০,০০০ লাইন...")
    wiki_en = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
    en_cnt = 0
    for item in wiki_en:
        for p in item.get('text', '').split('\n'):
            p = p.strip()
            if len(p) >= 35 and re.search(r'[a-zA-Z]', p):
                corpus_lines.append(p)
                en_cnt += 1
                if en_cnt >= 100000: break
        if en_cnt >= 100000: break
    print(f"     ✓ ইংরেজি উইকিপিডিয়া: {en_cnt:,} লাইন")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # SFT SOURCE ৪: বাংলা Alpaca-Orca ইনস্ট্রাকশন (৮০,০০০ জোড়া)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("  [৪/৬] বাংলা Alpaca-Orca ইনস্ট্রাকশন → ৮০,০০০ জোড়া...")
    try:
        alpaca_ds = load_dataset('BanglaLLM/bangla-alpaca-orca', split='train', streaming=True)
        alp_cnt = 0
        for row in alpaca_ds:
            inst = row.get('instruction', '').strip()
            inp  = row.get('input', '').strip()
            out  = row.get('output', '').strip()
            if inst and out:
                full_q = f"{inst} {inp}".strip()
                sft_lines.append(f"প্রশ্ন: {full_q} উত্তর: {out} <EOS>")
                corpus_lines.append(f"{full_q} {out}")
                alp_cnt += 1
                if alp_cnt >= 80000: break
        print(f"     ✓ বাংলা Alpaca-Orca: {alp_cnt:,} জোড়া")
    except Exception as e:
        print(f"     ⚠️ Alpaca ফলব্যাক: {e}")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # SFT SOURCE ৫: লোকাল ডোমেন JSONL (Digital Marketing, NCTB)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("  [৫/৬] লোকাল ডোমেন JSONL ডেটা লোড হচ্ছে...")
    jsonl_files = [
        'data/digital_marketing.jsonl',
        '/content/drive/MyDrive/digital_marketing.jsonl',
        'data/nctb_data.jsonl'
    ]
    for jf in jsonl_files:
        if os.path.exists(jf):
            with open(jf, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        d = json.loads(line.strip())
                        inst = d.get('instruction', '').strip()
                        out  = d.get('output', '').strip()
                        if inst and out:
                            corpus_lines.append(f"{inst} {out}")
                            sft_lines.append(f"প্রশ্ন: {inst} উত্তর: {out} <EOS>")
                    except: pass
            print(f"     ✓ লোড হয়েছে: {jf}")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # SFT SOURCE ৬: পাটিগণিত + ঐকিক নিয়ম + শতকরা (১,০০,০০০ লাইন)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("  [৬/৬] পাটিগণিত + জেনারেল গণিত → ১,০০,০০০ লাইন...")
    random.seed(42)
    math_types = ['add', 'sub', 'mul', 'div', 'percent', 'unitary', 'profit', 'en_math']
    for _ in range(100000):
        m = random.choice(math_types)
        if m == 'add':
            a, b = random.randint(5, 9999), random.randint(5, 9999)
            q   = f"{a} এর সাথে {b} যোগ করলে কত হয়?"
            ans = f"{a} + {b} = {a+b}।"
        elif m == 'sub':
            a, b = random.randint(50, 9999), random.randint(5, 4999)
            if a < b: a, b = b, a
            q   = f"{a} থেকে {b} বিয়োগ করলে কত থাকে?"
            ans = f"{a} - {b} = {a-b}।"
        elif m == 'mul':
            a, b = random.randint(2, 999), random.randint(2, 99)
            q   = f"{a} কে {b} দিয়ে গুণ করলে কত হবে?"
            ans = f"{a} × {b} = {a*b}।"
        elif m == 'div':
            d = random.randint(2, 30)
            q_val = d * random.randint(2, 100)
            q   = f"{q_val} কে {d} দিয়ে ভাগ করলে কত পাওয়া যায়?"
            ans = f"{q_val} ÷ {d} = {q_val//d}।"
        elif m == 'percent':
            base = random.choice([50, 100, 200, 500, 1000, 2000, 5000])
            rate = random.choice([5, 10, 15, 20, 25, 30, 50])
            val  = int(base * rate / 100)
            q   = f"{base} টাকার {rate}% কত?"
            ans = f"{base} × {rate}/100 = {val} টাকা।"
        elif m == 'unitary':
            n1 = random.randint(2, 8)
            up = random.randint(5, 50)
            c1 = n1 * up
            n2 = random.randint(9, 20)
            c2 = n2 * up
            q   = f"{n1}টি জিনিসের দাম {c1} টাকা হলে {n2}টির দাম কত?"
            ans = f"১টির দাম {c1}÷{n1}={up} টাকা। সুতরাং {n2}টির দাম {up}×{n2}={c2} টাকা।"
        elif m == 'profit':
            cp = random.randint(100, 5000)
            prof = random.randint(10, cp//2)
            sp = cp + prof
            q   = f"একটি জিনিস {cp} টাকায় কিনে {sp} টাকায় বিক্রি করলে লাভ কত?"
            ans = f"লাভ = {sp} - {cp} = {prof} টাকা।"
        else:  # en_math
            a, b = random.randint(2, 99), random.randint(2, 50)
            q   = f"What is {a} multiplied by {b}?"
            ans = f"{a} * {b} = {a*b}."

        corpus_lines.append(f"{q} {ans}")
        sft_lines.append(f"প্রশ্ন: {q} উত্তর: {ans} <EOS>")

    print(f"     ✓ গণিত লাইন: 1,00,000")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # শাফেল ও ফাইলে লেখা
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print("⚡ ডেটা শাফেল ও ফাইলে লেখা হচ্ছে...")
    random.shuffle(corpus_lines)
    random.shuffle(sft_lines)

    with open(corpus_path, 'w', encoding='utf-8') as f:
        for l in corpus_lines: f.write(l + '\n')

    with open(sft_data_path, 'w', encoding='utf-8') as f:
        for l in sft_lines: f.write(l + '\n')

    for old in ['data/corpus_tokens.bin', 'data/sft_data_tokens.bin']:
        if os.path.exists(old): os.remove(old)

    c_mb = os.path.getsize(corpus_path)   / (1024*1024)
    s_mb = os.path.getsize(sft_data_path) / (1024*1024)
    print("=" * 65)
    print(f"✓ CORPUS  (corpus.txt):   {len(corpus_lines):>10,} লাইন  ({c_mb:.1f} MB)")
    print(f"✓ SFT     (sft_data.txt): {len(sft_lines):>10,} লাইন  ({s_mb:.1f} MB)")
    print("=" * 65)

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # লাইভ স্টেপ ক্যালকুলেশন প্রিন্ট
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    BATCH          = 4
    GRAD_ACCUM     = 4
    BLOCK_SIZE     = 512
    AVG_TOKENS_LINE = 30
    TRAIN_SPLIT    = 0.9
    STAGE1_EPOCHS  = 2

    est_tokens      = len(corpus_lines) * AVG_TOKENS_LINE * TRAIN_SPLIT
    tokens_per_step = BATCH * GRAD_ACCUM * BLOCK_SIZE
    steps_per_epoch = int(est_tokens / tokens_per_step)
    total_steps     = steps_per_epoch * STAGE1_EPOCHS

    print(f"\n📊 স্টেজ-১ স্টেপ ক্যালকুলেশন:")
    print(f"   মোট corpus লাইন      : {len(corpus_lines):,}")
    print(f"   আনুমানিক মোট টোকেন   : {int(est_tokens):,}")
    print(f"   Tokens per Step       : {tokens_per_step:,}")
    print(f"   Steps per Epoch       : {steps_per_epoch:,}")
    print(f"   মোট স্টেপ (২ ইপক)    : {total_steps:,}")
    print(f"   আনুমানিক সময় (T4)    : ~{total_steps//3//60} মিনিট")


In [ ]:
# Step 5: ⚡ 10,000 Vocab ByteLevel BPE টোকেনাইজার তৈরি
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<|system|>', '<|user|>', '<|assistant|>', '<|math|>']
tok = Tokenizer(models.BPE(unk_token='<UNK>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True
)

print("⚡ ১০ লাখ লাইনের ডেটায় টোকেনাইজার ট্রেনিং শুরু...")
tok.train(['data/corpus.txt', 'data/sft_data.txt'], trainer)
tok.save('tokenizer.json')
print(f"✓ টোকেনাইজার প্রস্তুত! Vocab: {tok.get_vocab_size():,}")

In [ ]:
# Step 6: 🚀 [স্টেজ ১] প্রি-ট্রেনিং — ২ ইপক (~৬,৫৯০ স্টেপ)
import os, glob, time, math, torch
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

tokenizer       = Tokenizer.from_file('tokenizer.json')
dataset_stage1  = BengaliDataset(corpus_path='data/corpus.txt', tokenizer=tokenizer,
                                  block_size=GPTConfig.block_size, split_ratio=0.9)

tokens_per_step  = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
steps_per_epoch  = dataset_stage1.train_len // tokens_per_step
STAGE1_EPOCHS    = 2.0
stage1_max_iters = int(steps_per_epoch * STAGE1_EPOCHS)

print(f"📊 স্টেজ-১ স্টেপ হিসাব (প্রকৃত টোকেন থেকে):")
print(f"   মোট Train টোকেন  : {dataset_stage1.train_len:,}")
print(f"   Tokens per Step  : {tokens_per_step:,}")
print(f"   Steps per Epoch  : {steps_per_epoch:,}")
print(f"   মোট স্টেপ (২ ইপক): {stage1_max_iters:,}")
print(f"   আনুমানিক সময়    : ~{stage1_max_iters // 3 // 60} মিনিট (T4 @3.5 it/s)")

raw_model = GPT(GPTConfig).to(device)
try:
    model = torch.compile(raw_model)
    print('✓ torch.compile সক্রিয়!')
except:
    model = raw_model

optimizer = torch.optim.AdamW(raw_model.parameters(),
                               lr=GPTConfig.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler    = GradScaler()

# Auto-Resume
start_step = 1
ckpts = glob.glob(os.path.join(STAGE1_DIR, 'stage1_step_*.pt'))
if ckpts:
    def _s(p):
        try: return int(p.split('_step_')[-1].replace('.pt',''))
        except: return 0
    lc = sorted(ckpts, key=_s)[-1]
    ls = _s(lc)
    if 0 < ls < stage1_max_iters:
        raw_model.load_state_dict(torch.load(lc, map_location=device))
        start_step = ls + 1
        print(f'🔄 Auto-Resume: স্টেপ {start_step:,} থেকে শুরু')

def get_lr(it, mx, lr=3e-4, mlr=3e-5):
    w = 400
    if it < w: return lr * it / w
    if it > mx: return mlr
    d = (it-w)/(mx-w)
    return mlr + .5*(1+math.cos(math.pi*d))*(lr-mlr)

print('=' * 65)
print(f'🔥 [Stage 1] ট্রেনিং: {start_step:,} → {stage1_max_iters:,} স্টেপ')
print('=' * 65)

model.train()
optimizer.zero_grad(set_to_none=True)
t0 = time.time()

for step in range(start_step, stage1_max_iters + 1):
    lr = get_lr(step, stage1_max_iters, GPTConfig.learning_rate, GPTConfig.min_lr)
    for g in optimizer.param_groups: g['lr'] = lr

    acc_loss = 0.0
    for _ in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset_stage1.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _, loss = model(x, targets=y)
            loss    = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        acc_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if step % 250 == 0 or step == start_step:
        ep  = (step * tokens_per_step) / dataset_stage1.train_len
        ela = time.time() - t0
        spd = (step - start_step + 1) / ela if ela > 0 else 0
        eta = (stage1_max_iters - step) / spd / 60 if spd > 0 else 0
        print(f'[S1] Step {step:5d}/{stage1_max_iters} (Ep {ep:.2f}) | '
              f'Loss: {acc_loss:.4f} | LR: {lr:.2e} | '
              f'Speed: {spd:.2f} it/s | ETA: {eta:.1f} min')

    if step % 500 == 0 or step == stage1_max_iters:
        ck = os.path.join(STAGE1_DIR, f'stage1_step_{step}.pt')
        torch.save(raw_model.state_dict(), ck)

final1 = os.path.join(DRIVE_BASE_DIR, 'checkpoint_stage_1.pt')
torch.save(raw_model.state_dict(), final1)
print(f'🎉 স্টেজ ১ সম্পন্ন! → {final1}')

In [ ]:
# Step 7: 🎯 [স্টেজ ২] SFT ফাইন-টিউনিং — ৩ ইপক (~২২০ স্টেপ)
import os, glob, time, math, torch
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

s1_path = os.path.join(DRIVE_BASE_DIR, 'checkpoint_stage_1.pt')
raw_sft = GPT(GPTConfig).to(device)
if os.path.exists(s1_path):
    raw_sft.load_state_dict(torch.load(s1_path, map_location=device))
    print(f'📥 Stage 1 লোড: {s1_path}')
else:
    print('⚠️ Stage 1 চেকপয়েন্ট নেই, স্ক্র্যাচ থেকে শুরু!')

try:
    sft_model = torch.compile(raw_sft)
    print('✓ torch.compile সক্রিয়!')
except:
    sft_model = raw_sft

tokenizer      = Tokenizer.from_file('tokenizer.json')
dataset_stage2 = BengaliDataset(corpus_path='data/sft_data.txt', tokenizer=tokenizer,
                                 block_size=GPTConfig.block_size, split_ratio=0.9)

tokens_per_step  = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
steps_per_epoch  = max(1, dataset_stage2.train_len // tokens_per_step)
STAGE2_EPOCHS    = 3.0
stage2_max_iters = int(steps_per_epoch * STAGE2_EPOCHS)

print(f"📊 স্টেজ-২ স্টেপ হিসাব:")
print(f"   Train টোকেন      : {dataset_stage2.train_len:,}")
print(f"   Tokens per Step  : {tokens_per_step:,}")
print(f"   Steps per Epoch  : {steps_per_epoch:,}")
print(f"   মোট স্টেপ (৩ ইপক): {stage2_max_iters:,}")

optimizer = torch.optim.AdamW(raw_sft.parameters(),
                               lr=GPTConfig.sft_learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler    = GradScaler()

start_step_sft = 1
sft_ckpts = glob.glob(os.path.join(STAGE2_DIR, 'stage2_step_*.pt'))
if sft_ckpts:
    def _ss(p):
        try: return int(p.split('_step_')[-1].replace('.pt',''))
        except: return 0
    lc2 = sorted(sft_ckpts, key=_ss)[-1]
    ls2 = _ss(lc2)
    if 0 < ls2 < stage2_max_iters:
        raw_sft.load_state_dict(torch.load(lc2, map_location=device))
        start_step_sft = ls2 + 1
        print(f'🔄 SFT Auto-Resume: স্টেপ {start_step_sft:,} থেকে')

def get_sft_lr(it, mx, lr=1e-4, mlr=1e-5):
    w = min(100, mx//10)
    if it < w: return lr * it / w
    if it > mx: return mlr
    d = (it-w)/max(1, mx-w)
    return mlr + .5*(1+math.cos(math.pi*d))*(lr-mlr)

print('=' * 65)
print(f'🔥 [Stage 2] SFT: {start_step_sft:,} → {stage2_max_iters:,} স্টেপ')
print('=' * 65)

sft_model.train()
optimizer.zero_grad(set_to_none=True)
t0 = time.time()

for step in range(start_step_sft, stage2_max_iters + 1):
    lr = get_sft_lr(step, stage2_max_iters, GPTConfig.sft_learning_rate, GPTConfig.sft_min_lr)
    for g in optimizer.param_groups: g['lr'] = lr

    acc_loss = 0.0
    for _ in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset_stage2.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _, loss = sft_model(x, targets=y)
            loss    = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        acc_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_sft.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if step % 25 == 0 or step == start_step_sft or step == stage2_max_iters:
        ep  = (step * tokens_per_step) / dataset_stage2.train_len
        ela = time.time() - t0
        spd = (step - start_step_sft + 1) / ela if ela > 0 else 0
        print(f'[S2] Step {step:5d}/{stage2_max_iters} (Ep {ep:.2f}) | '
              f'Loss: {acc_loss:.4f} | LR: {lr:.2e} | Speed: {spd:.2f} it/s')

    if step % 100 == 0 or step == stage2_max_iters:
        ck = os.path.join(STAGE2_DIR, f'stage2_step_{step}.pt')
        torch.save(raw_sft.state_dict(), ck)

final2 = os.path.join(DRIVE_BASE_DIR, 'checkpoint_stage_2_final.pt')
torch.save(raw_sft.state_dict(), final2)
print(f'🎉 চূড়ান্ত মডেল প্রস্তুত! → {final2}')

In [ ]:
# Step 8: 💬 চ্যাটবট টেস্ট
final2 = os.path.join(DRIVE_BASE_DIR, 'checkpoint_stage_2_final.pt')
chat_model = GPT(GPTConfig).to(device)
chat_model.load_state_dict(torch.load(final2, map_location=device))
chat_model.eval()
tokenizer = Tokenizer.from_file('tokenizer.json')

# ✏️ এখানে আপনার প্রশ্ন লিখুন:
user_question = 'ডিজিটাল মার্কেটিং কী?'

prompt = f'প্রশ্ন: {user_question} উত্তর:'
enc    = tokenizer.encode(prompt)
ids    = enc.ids if hasattr(enc, 'ids') else enc
inp    = torch.tensor([ids], dtype=torch.long, device=device)
eos_id = tokenizer.token_to_id('<EOS>')

with torch.no_grad():
    out = chat_model.generate(inp, max_new_tokens=200,
                               temperature=0.7, top_k=40,
                               repetition_penalty=1.25, eos_id=eos_id)

reply = tokenizer.decode(out[0].cpu().tolist()).split('<EOS>')[0].strip()
print('=' * 60)
print(reply)
print('=' * 60)